# PCA-50 Full Test Error

This notebook only analyzes eval-adoption runs whose result-group name contains `pca50`, focusing on non-null `full_test_error`.


In [ ]:
from pathlib import Path
import re

import pandas as pd
import plotly.express as px

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)


In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'results').exists() and (ROOT.parent / 'results').exists():
    ROOT = ROOT.parent

RESULTS_ROOT = ROOT / 'results'
RESULTS_ROOT


In [ ]:
EVAL_ADOPTION_RE = re.compile(
    r'^(?:eval-adoption-)?absolute_accuracy_decay__(?P<perturbation_type>.+?)__(?:(?P<probe_control>none|randomization|permutation)__)?L(?P<layer>\d+)$'
)

def load_eval_adoption_metrics(results_root: Path) -> pd.DataFrame:
    rows = []
    for metrics_path in results_root.rglob('metrics.csv'):
        rel = metrics_path.relative_to(results_root)
        parts = rel.parts
        if len(parts) < 8:
            continue

        probe_idx = next((i for i, part in enumerate(parts) if EVAL_ADOPTION_RE.match(part) or re.search(r'(?:_L|__L)\d+$', part)), None)
        if probe_idx is None:
            continue

        probe_name = parts[probe_idx]
        tail = parts[probe_idx + 1 :]
        if len(tail) == 7:
            model_name, encoding, control_task, sample_size, seed, fold, _ = tail
            status = 'completed'
        elif len(tail) == 8:
            model_name, encoding, control_task, sample_size, seed, fold, status, _ = tail
        else:
            continue

        result_group = '/'.join(parts[:probe_idx])
        if 'pca50' not in result_group.lower():
            continue

        match = EVAL_ADOPTION_RE.match(probe_name)
        if not match:
            continue

        frame = pd.read_csv(metrics_path)
        if frame.empty:
            continue

        values = frame.iloc[0].to_dict()
        values.update({
            'result_group': result_group,
            'probe_name': probe_name,
            'model_name': model_name,
            'encoding': encoding,
            'control_task': control_task,
            'sample_size': sample_size,
            'seed': seed,
            'fold': fold,
            'layer': int(match.group('layer')),
            'perturbation_type': match.group('perturbation_type'),
            'probe_control': (match.group('probe_control') or control_task.lower()),
            'status': status,
            'metrics_path': str(metrics_path),
            'group_mtime': (results_root / result_group).stat().st_mtime,
        })
        rows.append(values)

    if not rows:
        raise RuntimeError(f'No pca50 eval-adoption metrics found under {results_root}')

    df = pd.DataFrame(rows)
    if 'full test error' not in df.columns:
        raise RuntimeError('Expected `full test error` in metrics.csv files')

    df = df.rename(columns={'full test error': 'full_test_error'})
    df['full_test_error'] = pd.to_numeric(df['full_test_error'], errors='coerce')
    df = df[df['full_test_error'].notna()].copy()
    if df.empty:
        raise RuntimeError('PCA-50 runs were found, but none have non-null full_test_error yet')
    return df

metrics = load_eval_adoption_metrics(RESULTS_ROOT)
metrics[['result_group', 'probe_name', 'control_task', 'probe_control', 'layer', 'full_test_error']].head()


In [ ]:
latest_group = metrics.sort_values('group_mtime')['result_group'].iloc[-1]
latest = metrics[metrics['result_group'] == latest_group].copy()
latest = latest[latest['probe_control'].isin(['none', 'randomization'])].copy()
latest = latest.sort_values(['probe_control', 'perturbation_type', 'layer']).reset_index(drop=True)

print('Latest PCA-50 result group:', latest_group)
print('Rows:', len(latest))
print('Controls:', sorted(latest['probe_control'].unique()))
print('Perturbation types:', sorted(latest['perturbation_type'].unique()))


In [ ]:
best_by_group = (
    latest.sort_values('full_test_error')
    .groupby(['probe_control', 'perturbation_type'], as_index=False)
    .first()[['probe_control', 'perturbation_type', 'layer', 'full_test_error', 'probe_name']]
    .sort_values(['probe_control', 'full_test_error', 'perturbation_type'])
)
best_by_group


In [ ]:
summary = (
    latest.groupby(['probe_control', 'perturbation_type'])['full_test_error']
    .agg(['count', 'min', 'mean', 'median', 'max'])
    .reset_index()
    .sort_values(['probe_control', 'mean', 'perturbation_type'])
)
summary


In [ ]:
fig = px.line(
    latest.sort_values('layer'),
    x='layer',
    y='full_test_error',
    color='perturbation_type',
    facet_col='probe_control',
    markers=True,
    category_orders={'probe_control': ['none', 'randomization']},
    title='PCA-50 full_test_error by layer, control task, and perturbation type',
)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace('probe_control=', 'control=')))
fig.update_xaxes(title_text='layer')
fig.update_yaxes(title_text='full_test_error')
fig.show()


In [ ]:
fig = px.scatter(
    latest,
    x='layer',
    y='perturbation_type',
    color='full_test_error',
    facet_col='probe_control',
    category_orders={'probe_control': ['none', 'randomization']},
    color_continuous_scale='Viridis_r',
    title='PCA-50 full_test_error heatmap view by control and perturbation type',
)
fig.update_traces(marker={'size': 14, 'symbol': 'square'})
fig.for_each_annotation(lambda a: a.update(text=a.text.replace('probe_control=', 'control=')))
fig.update_xaxes(title_text='layer')
fig.update_yaxes(title_text='perturbation_type')
fig.show()
